In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# تحميل البيانات
train_data = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')

print(f"Training data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# الميزات الرقمية عالية التأثير
important_numeric_features = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
    '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd',
    'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', 'OpenPorchSF',
    'LotArea', 'LotFrontage', '2ndFlrSF', 'BsmtUnfSF', 'BsmtFinSF2'
]

# الميزات الفئوية عالية التأثير
important_categorical_features = [
    'Neighborhood', 'MSZoning', 'Street', 'Alley', 'LotShape',
    'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1',
    'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st',
    'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PavedDrive', 'SaleType', 'SaleCondition'
]

# جميع الميزات المهمة
all_features = important_numeric_features + important_categorical_features

def advanced_preprocessing(train_df, test_df, features):
    """
    معالجة متقدمة للبيانات تشمل التعامل مع القيم المفقودة والترميز
    """
    # نسخ البيانات
    train_processed = train_df.copy()
    test_processed = test_df.copy()
    
    # التأكد من وجود الميزات في كلا المجموعتين
    available_features = [f for f in features if f in train_df.columns and f in test_df.columns]
    
    # فصل الميزات الرقمية والفئوية
    numeric_features = [f for f in available_features if train_df[f].dtype in ['int64', 'float64']]
    categorical_features = [f for f in available_features if train_df[f].dtype == 'object']
    
    print(f"Processing {len(numeric_features)} numeric features and {len(categorical_features)} categorical features")
    
    # معالجة الميزات الرقمية
    for feature in numeric_features:
        # استخدام الوسيط للقيم المفقودة
        median_val = train_processed[feature].median()
        train_processed[feature].fillna(median_val, inplace=True)
        test_processed[feature].fillna(median_val, inplace=True)
    
    # معالجة الميزات الفئوية
    for feature in categorical_features:
        # استخدام القيمة الأكثر تكراراً للقيم المفقودة
        most_frequent = train_processed[feature].mode()[0] if not train_processed[feature].mode().empty else 'Missing'
        train_processed[feature].fillna(most_frequent, inplace=True)
        test_processed[feature].fillna(most_frequent, inplace=True)
        
        # ترميز الميزات الفئوية
        le = LabelEncoder()
        # دمج البيانات لضمان وجود جميع الفئات
        combined = pd.concat([train_processed[feature], test_processed[feature]], axis=0)
        le.fit(combined)
        
        train_processed[feature] = le.transform(train_processed[feature])
        test_processed[feature] = le.transform(test_processed[feature])
    
    return train_processed[available_features], test_processed[available_features]

# تحضير البيانات
X_train_processed, X_test_processed = advanced_preprocessing(train_data, test_data, all_features)
y_train = train_data['SalePrice']

def create_advanced_features(df):
    """
    إنشاء ميزات جديدة لتحسين أداء النموذج
    """
    df = df.copy()
    
    # المساحة الإجمالية للمنزل
    df['TotalSF'] = df.get('TotalBsmtSF', 0) + df.get('1stFlrSF', 0) + df.get('2ndFlrSF', 0)
    
    # عمر المنزل
    df['HouseAge'] = df.get('YrSold', 2023) - df.get('YearBuilt', 0)
    
    # هل تم تجديد المنزل؟
    df['IsRemodeled'] = (df.get('YearRemodAdd', 0) != df.get('YearBuilt', 0)).astype(int)
    
    # عدد الحمامات الإجمالي
    df['TotalBath'] = df.get('FullBath', 0) + 0.5 * df.get('HalfBath', 0) + \
                      df.get('BsmtFullBath', 0) + 0.5 * df.get('BsmtHalfBath', 0)
    
    # المساحة الخارجية الإجمالية
    df['TotalPorchSF'] = df.get('OpenPorchSF', 0) + df.get('EnclosedPorch', 0) + \
                         df.get('3SsnPorch', 0) + df.get('ScreenPorch', 0)
    
    # جودة عامة مجمعة
    df['OverallGrade'] = df.get('OverallQual', 0) * df.get('OverallCond', 0)
    
    # المساحة المعيشية الإجمالية مع القبو
    df['TotalLivArea'] = df.get('GrLivArea', 0) + df.get('TotalBsmtSF', 0)
    
    return df

# تطبيق هندسة الميزات
X_train_enhanced = create_advanced_features(X_train_processed)
X_test_enhanced = create_advanced_features(X_test_processed)

# تقسيم البيانات للتحقق من الأداء
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_enhanced, y_train, test_size=0.2, random_state=42
)

# نموذج الغابة العشوائية مع معاملات محسنة
advanced_model = RandomForestRegressor(
    n_estimators=500,        # زيادة عدد الأشجار
    max_depth=25,            # زيادة العمق
    min_samples_split=5,     # تقليل العينات المطلوبة للانقسام
    min_samples_leaf=2,      # تقليل العينات في الورقة
    max_features='sqrt',     # استخدام الجذر التربيعي لعدد الميزات
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# تدريب النموذج
print("Training advanced model...")
advanced_model.fit(X_train, y_train_split)

# التحقق من الأداء
val_predictions = advanced_model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_predictions)
print(f"Validation MAE: {val_mae:,.0f}")

# التحقق المتقاطع
cv_scores = cross_val_score(advanced_model, X_train_enhanced, y_train, 
                           scoring='neg_mean_absolute_error', cv=5)
print(f"Cross-Validation MAE: {-cv_scores.mean():,.0f} (+/- {cv_scores.std() * 2:,.0f})")

# تحليل أهمية الميزات
feature_importance = pd.DataFrame({
    'feature': X_train_enhanced.columns,
    'importance': advanced_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20))

# رسم أهمية الميزات
plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importance')
plt.tight_layout()
plt.show()

# تدريب النموذج النهائي على جميع بيانات التدريب
print("Training final model on all data...")
final_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_enhanced, y_train)

# التنبؤ ببيانات الاختبار
print("Making predictions on test data...")
test_predictions = final_model.predict(X_test_enhanced)

# إنشاء ملف التقديم
submission = pd.DataFrame({
    'Id': test_data['Id'],
    'SalePrice': test_predictions
})

# حفظ الملف
submission.to_csv('advanced_submission.csv', index=False)
print("Advanced submission file created!")

# عرض إحصائيات التنبؤات
print(f"\nPredictions Statistics:")
print(f"Min Price: ${submission['SalePrice'].min():,.0f}")
print(f"Max Price: ${submission['SalePrice'].max():,.0f}")
print(f"Mean Price: ${submission['SalePrice'].mean():,.0f}")
print(f"Median Price: ${submission['SalePrice'].median():,.0f}")

# إذا أردت تجربة XGBoost (يتطلب تثبيت xgboost)
try:
    from xgboost import XGBRegressor
    
    xgb_model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(X_train_enhanced, y_train)
    xgb_preds = xgb_model.predict(X_test_enhanced)
    
    # يمكنك تجربة متوسط تنبؤات النماذج
    ensemble_predictions = (test_predictions + xgb_preds) / 2
    
    # حفظ التقديم المجمع
    ensemble_submission = pd.DataFrame({
        'Id': test_data['Id'],
        'SalePrice': ensemble_predictions
    })
    ensemble_submission.to_csv('ensemble_submission.csv', index=False)
    print("Ensemble submission file created!")
    
except ImportError:
    print("XGBoost not available, using Random Forest only")
    